# 03 - Finetune YOLOPv2 (Colab GPU)

This notebook fine-tunes YOLOPv2 on your mixed dataset (CARLA + pseudo-labeled real) with:
- deterministic train/val/test split persisted on Google Drive
- checkpointing + automatic resume after kernel restarts
- early stopping (patience=10)
- LR decay + AdamW
- training curves saved and plotted consistently across multiple sessions

References related to this project:
- Pseudo-labeling: Caine et al., arXiv 2021
- Domain adaptation: arXiv 2020


## Section 1 — Setup

Installs dependencies, clones `CAIC-AD/YOLOPv2` (primary), mounts Google Drive, and defines all paths.
Falls back to `hustvl/YOLOP` if training configs are missing in the primary repo.


In [1]:
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --quiet
%pip install -q opencv-python pandas numpy matplotlib tqdm onnx onnxruntime onnxruntime-gpu yacs pyyaml --quiet
!pip install prefetch_generator --quiet

In [2]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: request to https://colab.research.google.com/tun/m/credentials-propagation/gpu-t4-s-kkb-euw4b0-222sl0m41mbmk?authtype=dfs_ephemeral&version=2&dryrun=true&propagate=true&record=false&authuser=0 failed, reason: 

In [4]:
import os
import sys
import json
import time
import math
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Primary: CAIC-AD/YOLOPv2 (expected by the project spec).
PRIMARY_REPO = Path('/content/YOLOPv2')
if not PRIMARY_REPO.exists():
    !git clone https://github.com/CAIC-AD/YOLOPv2.git /content/YOLOPv2

# Fallback: hustvl/YOLOP (training-ready if primary repo lacks configs).
FALLBACK_REPO = Path('/content/YOLOP')
if not FALLBACK_REPO.exists():
    !git clone https://github.com/hustvl/YOLOP.git /content/YOLOP

def list_cfgs(repo: Path):
    cfg_dir = repo / 'cfg'
    if not cfg_dir.exists():
        return []
    return sorted(cfg_dir.glob('*.yaml'))

def has_training_lib(repo: Path) -> bool:
    return (repo / 'lib').exists()
def find_repo_and_cfg(primary: Path, fallback: Path):
    for repo in (primary, fallback):
        cfgs = list_cfgs(repo)
        if cfgs and has_training_lib(repo):
            return repo, cfgs
    return primary, []

REPO_ROOT, CFG_CANDIDATES = find_repo_and_cfg(PRIMARY_REPO, FALLBACK_REPO)
if not CFG_CANDIDATES or not has_training_lib(REPO_ROOT):
    raise FileNotFoundError(
        f'No training-ready YOLOP repo found. '
        f'Primary cfgs: {list_cfgs(PRIMARY_REPO)}, lib={has_training_lib(PRIMARY_REPO)} | '
        f'Fallback cfgs: {list_cfgs(FALLBACK_REPO)}, lib={has_training_lib(FALLBACK_REPO)}'
    )

preferred = [p for p in CFG_CANDIDATES if 'yolop' in p.stem.lower()]
CFG_PATH = preferred[0] if preferred else CFG_CANDIDATES[0]
if REPO_ROOT != PRIMARY_REPO:
    print('Primary repo missing training configs; using fallback:', REPO_ROOT)

PROJECT_ROOT = Path('/content/drive/MyDrive/sdcar-perception')

CARLA_MANIFEST = PROJECT_ROOT / 'data' / 'carla' / 'manifest.csv'
PSEUDO_MANIFEST = PROJECT_ROOT / 'data' / 'pseudo' / 'manifest.csv'
MIXED_MANIFEST = PROJECT_ROOT / 'data' / 'yolopv2_finetune' / 'manifest.csv'  # optional consolidated manifest

RUN_DIR = PROJECT_ROOT / 'runs' / 'yolopv2_finetune'
CKPT_DIR = RUN_DIR / 'checkpoints'
EXPORT_DIR = RUN_DIR / 'exports'
RUN_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 640
BATCH_SIZE = 8
NUM_EPOCHS = 100
PATIENCE = 10
SEED = 1337

# Ensure repo is importable
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('REPO_ROOT:', REPO_ROOT)
print('CFG_PATH:', CFG_PATH)
print('RUN_DIR:', RUN_DIR)

# Sanity check (should show lib/, cfg/)
!ls -la {REPO_ROOT}

FileNotFoundError: No training-ready YOLOP repo found. Primary cfgs: [], lib=False | Fallback cfgs: [], lib=True

## Section 2 — Dataset class

Reads your manifest(s) and loads:
- image (resized to 640×640)
- YOLO labels (`cls cx cy w h`, normalized)
- lane and drivable binary masks
- domain label (`is_real` or `domain`)

No extra augmentation is applied here (your dataset is already augmented).


In [ ]:
def seed_everything(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)


def _read_yolo_file(path: Path):
    # YOLO label format: cls cx cy w h (all normalized)
    if not path.exists():
        return np.zeros((0, 5), dtype=np.float32)
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls_id, cx, cy, w, h = parts
        rows.append([float(cls_id), float(cx), float(cy), float(w), float(h)])
    if not rows:
        return np.zeros((0, 5), dtype=np.float32)
    return np.asarray(rows, dtype=np.float32)


def _binmask_to_2ch(mask01: np.ndarray) -> torch.Tensor:
    """Convert binary mask (H,W) to 2-class one-hot tensor (2,H,W).

    YOLOP's segmentation loss expects 2-channel targets.
    channel0 = background, channel1 = foreground.
    """
    m = (mask01 > 0).astype(np.float32)
    bg = 1.0 - m
    fg = m
    out = np.stack([bg, fg], axis=0)
    return torch.from_numpy(out)


class MixedPerceptionDataset(Dataset):
    def __init__(self, manifest_paths, image_size=640):
        dfs = [pd.read_csv(mp) for mp in manifest_paths]
        self.df = pd.concat(dfs, ignore_index=True)
        self.image_size = int(image_size)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = cv2.imread(str(row['image_path']))
        if img is None:
            raise FileNotFoundError(f"Failed to read image: {row['image_path']}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        lane = cv2.imread(str(row['lane_mask_path']), cv2.IMREAD_GRAYSCALE)
        drivable = cv2.imread(str(row['drivable_mask_path']), cv2.IMREAD_GRAYSCALE)
        if lane is None:
            lane = np.zeros((self.image_size, self.image_size), dtype=np.uint8)
        if drivable is None:
            drivable = np.zeros((self.image_size, self.image_size), dtype=np.uint8)

        lane = cv2.resize(lane, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
        drivable = cv2.resize(drivable, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)

        # YOLOP expects 2-channel seg targets
        lane_t = _binmask_to_2ch(lane)
        drivable_t = _binmask_to_2ch(drivable)

        labels = _read_yolo_file(Path(row['label_path']))  # [cls, cx, cy, w, h]
        labels_t = torch.from_numpy(labels) if labels.size else torch.zeros((0, 5), dtype=torch.float32)

        if 'is_real' in row:
            domain = float(row['is_real'])
        else:
            d = row.get('domain', 0)
            if isinstance(d, str):
                domain = 1.0 if d.lower() == 'real' else 0.0
            else:
                domain = float(d)
        domain_t = torch.tensor([domain], dtype=torch.float32)

        return {
            'image': img_t,
            'labels': labels_t,
            'lane_mask': lane_t,
            'drivable_mask': drivable_t,
            'domain_label': domain_t,
        }


def collate_fn(batch):
    return {
        'image': torch.stack([b['image'] for b in batch], dim=0),
        'labels': [b['labels'] for b in batch],
        'lane_mask': torch.stack([b['lane_mask'] for b in batch], dim=0),
        'drivable_mask': torch.stack([b['drivable_mask'] for b in batch], dim=0),
        'domain_label': torch.stack([b['domain_label'] for b in batch], dim=0),
    }


manifests = [MIXED_MANIFEST] if MIXED_MANIFEST.exists() else [p for p in [CARLA_MANIFEST, PSEUDO_MANIFEST] if p.exists()]
if not manifests:
    raise FileNotFoundError('No manifests found. Expected MIXED_MANIFEST or CARLA+PSEUDO manifests.')

full_ds = MixedPerceptionDataset(manifests, image_size=IMAGE_SIZE)
print('Total samples:', len(full_ds))
print('Using manifests:', manifests)


Total samples: 12928
Using manifests: [PosixPath('/content/drive/MyDrive/sdcar-perception/data/yolopv2_finetune/manifest.csv')]


## Section 3 — Train/Val/Test split (persisted)

Creates a deterministic 80/10/10 split and saves it to Drive so it stays identical across sessions.


In [7]:
from torch.utils.data import Subset

SPLIT_PATH = RUN_DIR / 'split_indices.pt'

df_all = full_ds.df.reset_index(drop=True)

def get_is_real(row):
    if 'is_real' in row:
        try:
            return int(row['is_real'])
        except Exception:
            pass
    d = row.get('domain', 0)
    if isinstance(d, str):
        return 1 if d.lower() == 'real' else 0
    return int(d)

if SPLIT_PATH.exists():
    split = torch.load(SPLIT_PATH, map_location='cpu')
    train_idx, val_idx, test_idx = split['train'], split['val'], split['test']
else:
    rng = np.random.default_rng(SEED)
    train_idx, val_idx, test_idx = [], [], []

    for is_real in [0, 1]:
        idxs = [int(i) for i, row in df_all.iterrows() if get_is_real(row) == is_real]
        rng.shuffle(idxs)
        n = len(idxs)
        n_train = int(0.8 * n)
        n_val = int(0.1 * n)
        train_idx += idxs[:n_train]
        val_idx += idxs[n_train:n_train + n_val]
        test_idx += idxs[n_train + n_val:]

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    split = {'train': train_idx, 'val': val_idx, 'test': test_idx, 'seed': SEED}
    torch.save(split, SPLIT_PATH)

train_ds = Subset(full_ds, train_idx)
val_ds = Subset(full_ds, val_idx)
test_ds = Subset(full_ds, test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

print('Split sizes:', {'train': len(train_ds), 'val': len(val_ds), 'test': len(test_ds)})


Split sizes: {'train': 10342, 'val': 1292, 'test': 1294}


## Section 4 — Model assembly

Loads YOLOPv2 from the official checkpoint and builds optimizer + LR decay.
This section supports resuming from `last.pt`.


In [3]:
import yaml
from lib.config import cfg as yolop_cfg
from lib.models import get_net
from lib.core.loss import get_loss

# YOLOPv2 weights (community mirror) to initialize training
OFFICIAL_CKPT = CKPT_DIR / 'yolopv2_official.pt'
if not OFFICIAL_CKPT.exists():
    !wget -O {OFFICIAL_CKPT} https://huggingface.co/nicehuster/yolopv2/resolve/main/yolopv2.pt

if CFG_PATH is None or not Path(CFG_PATH).exists():
    raise FileNotFoundError(f'CFG_PATH not set or missing: {CFG_PATH}')
print('Using cfg:', CFG_PATH)

yolop_cfg.defrost()
yolop_cfg.merge_from_file(str(CFG_PATH))
yolop_cfg.MODEL.IMAGE_SIZE = [IMAGE_SIZE, IMAGE_SIZE]
yolop_cfg.TRAIN.END_EPOCH = NUM_EPOCHS
# We do not rely on YOLOP's built-in dataset pipeline, only model+loss.
yolop_cfg.freeze()

# Build model
model = get_net(yolop_cfg)

# Load YOLOPv2 checkpoint into YOLOP model (strict=False) with prefix cleanup
ckpt_raw = torch.load(OFFICIAL_CKPT, map_location='cpu')
state = ckpt_raw.get('state_dict', ckpt_raw) if isinstance(ckpt_raw, dict) else ckpt_raw

clean = {}
for k, v in state.items():
    nk = k
    if nk.startswith('module.'):
        nk = nk[len('module.'):]
    if nk.startswith('model.'):
        nk = nk[len('model.'):]
    clean[nk] = v

missing, unexpected = model.load_state_dict(clean, strict=False)
print('Loaded weights with strict=False')
print(' - missing keys:', len(missing))
print(' - unexpected keys:', len(unexpected))

model = model.to(device)

# YOLOP multi-head loss (detection + drivable seg + lane seg)
criterion = get_loss(yolop_cfg, device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

print('YOLOP model ready on', device)

ModuleNotFoundError: No module named 'lib'

## Section 5 — Training loop (100 epochs, resumable)

Adds:
- persistent `history.csv`
- `last.pt` and `best.pt` checkpoints
- early stopping with patience=10 (validation total loss)
- gradient clipping
- automatic resume after kernel restart


In [ ]:
HISTORY_CSV = RUN_DIR / 'history.csv'
LAST_CKPT = CKPT_DIR / 'last.pt'
BEST_CKPT = CKPT_DIR / 'best.pt'


def build_det_targets(labels_list, device):
    # YOLOP expects det targets shaped (n,6): [img_idx, cls, cx, cy, w, h]
    out = []
    for i, lab in enumerate(labels_list):
        if lab is None or lab.numel() == 0:
            continue
        lab = lab.to(device)
        img_idx = torch.full((lab.shape[0], 1), i, device=device, dtype=lab.dtype)
        t = torch.cat([img_idx, lab[:, 0:1], lab[:, 1:]], dim=1)
        out.append(t)
    if not out:
        return torch.zeros((0, 6), device=device)
    return torch.cat(out, dim=0)


def forward_unpack(model, images):
    # YOLOP forward: det_out, da_seg_out, ll_seg_out
    out = model(images)
    if isinstance(out, (list, tuple)) and len(out) >= 3:
        det_out, da_seg_out, ll_seg_out = out[0], out[1], out[2]
        return det_out, da_seg_out, ll_seg_out
    raise RuntimeError(f'Unexpected model output type/shape: {type(out)}')


def eval_one_epoch(model, loader):
    model.eval()
    totals = {'total': 0.0, 'lbox': 0.0, 'lobj': 0.0, 'lcls': 0.0, 'lseg_da': 0.0, 'lseg_ll': 0.0, 'liou_ll': 0.0}
    n_batches = 0

    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            da_gt = batch['drivable_mask'].to(device)  # (B,2,H,W)
            ll_gt = batch['lane_mask'].to(device)      # (B,2,H,W)
            det_t = build_det_targets(batch['labels'], device)

            det_out, da_out, ll_out = forward_unpack(model, images)

            # YOLOP loss expects:
            # predictions = [[P3,P4,P5], da_seg_logits, ll_seg_logits]
            # targets = [det_targets, da_targets, ll_targets]
            # shapes used for padding removal in ll IoU term; we have no letterbox padding => 0
            bsz = images.shape[0]
            shapes = [((IMAGE_SIZE, IMAGE_SIZE), ((1.0, 1.0), (0, 0))) for _ in range(bsz)]

            loss, parts = criterion([det_out, da_out, ll_out], [det_t, da_gt, ll_gt], shapes, model)
            lbox, lobj, lcls, lseg_da, lseg_ll, liou_ll, ltotal = parts

            totals['total'] += float(ltotal)
            totals['lbox'] += float(lbox)
            totals['lobj'] += float(lobj)
            totals['lcls'] += float(lcls)
            totals['lseg_da'] += float(lseg_da)
            totals['lseg_ll'] += float(lseg_ll)
            totals['liou_ll'] += float(liou_ll)
            n_batches += 1

    if n_batches == 0:
        return {k: math.inf for k in totals}
    return {k: v / n_batches for k, v in totals.items()}


# Restore history
history = []
if HISTORY_CSV.exists():
    try:
        history = pd.read_csv(HISTORY_CSV).to_dict('records')
    except Exception:
        history = []

# Resume state
start_epoch = 1
best_val = math.inf
patience_left = PATIENCE

if LAST_CKPT.exists():
    ck = torch.load(LAST_CKPT, map_location=device)
    model.load_state_dict(ck['model'], strict=False)
    optimizer.load_state_dict(ck['optimizer'])
    scheduler.load_state_dict(ck['scheduler'])
    if ck.get('scaler') is not None:
        scaler.load_state_dict(ck['scaler'])

    start_epoch = int(ck.get('epoch', 0)) + 1
    best_val = float(ck.get('best_val', best_val))
    patience_left = int(ck.get('patience_left', PATIENCE))

print('Resume:', {'start_epoch': start_epoch, 'best_val': best_val, 'patience_left': patience_left})


for epoch in range(start_epoch, NUM_EPOCHS + 1):
    model.train()

    running = {'total': 0.0, 'lbox': 0.0, 'lobj': 0.0, 'lcls': 0.0, 'lseg_da': 0.0, 'lseg_ll': 0.0, 'liou_ll': 0.0}
    n_batches = 0

    for step, batch in enumerate(tqdm(train_loader, desc=f'Train epoch {epoch}'), start=1):
        images = batch['image'].to(device)
        da_gt = batch['drivable_mask'].to(device)
        ll_gt = batch['lane_mask'].to(device)
        det_t = build_det_targets(batch['labels'], device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            det_out, da_out, ll_out = forward_unpack(model, images)

            bsz = images.shape[0]
            shapes = [((IMAGE_SIZE, IMAGE_SIZE), ((1.0, 1.0), (0, 0))) for _ in range(bsz)]

            loss, parts = criterion([det_out, da_out, ll_out], [det_t, da_gt, ll_gt], shapes, model)
            lbox, lobj, lcls, lseg_da, lseg_ll, liou_ll, ltotal = parts

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        scaler.step(optimizer)
        scaler.update()

        running['total'] += float(ltotal)
        running['lbox'] += float(lbox)
        running['lobj'] += float(lobj)
        running['lcls'] += float(lcls)
        running['lseg_da'] += float(lseg_da)
        running['lseg_ll'] += float(lseg_ll)
        running['liou_ll'] += float(liou_ll)
        n_batches += 1

        if step % 25 == 0:
            print(
                f"epoch={epoch} step={step} "
                f"L_total={running['total']/n_batches:.4f} "
                f"box={running['lbox']/n_batches:.4f} obj={running['lobj']/n_batches:.4f} cls={running['lcls']/n_batches:.4f} "
                f"da={running['lseg_da']/n_batches:.4f} ll={running['lseg_ll']/n_batches:.4f} liou_ll={running['liou_ll']/n_batches:.4f} "
                f"lr={optimizer.param_groups[0]['lr']:.2e}"
            )

    train_metrics = {k: v / max(n_batches, 1) for k, v in running.items()}
    val_metrics = eval_one_epoch(model, val_loader)

    scheduler.step()

    row = {
        'epoch': epoch,
        'lr': optimizer.param_groups[0]['lr'],
        **{f'train_{k}': v for k, v in train_metrics.items()},
        **{f'val_{k}': v for k, v in val_metrics.items()},
        'best_val_total': best_val,
        'patience_left': patience_left,
        'time': time.time(),
    }
    history.append(row)
    pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)

    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict() if scaler is not None else None,
        'best_val': best_val,
        'patience_left': patience_left,
        'image_size': IMAGE_SIZE,
    }, LAST_CKPT)

    improved = val_metrics['total'] < best_val
    if improved:
        best_val = val_metrics['total']
        patience_left = PATIENCE
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler': scaler.state_dict() if scaler is not None else None,
            'best_val': best_val,
            'image_size': IMAGE_SIZE,
        }, BEST_CKPT)
        print(f'New best: val_total={best_val:.4f} @ epoch={epoch}')
    else:
        patience_left -= 1
        print(f'No improvement. patience_left={patience_left}')

    if epoch % 10 == 0:
        torch.save({'epoch': epoch, 'model': model.state_dict(), 'best_val': best_val}, CKPT_DIR / f'epoch_{epoch}.pt')

    if patience_left <= 0:
        print(f'Early stopping at epoch {epoch} (patience={PATIENCE}).')
        break

print('Done. History saved to', HISTORY_CSV)
print('Best val total:', best_val)


## Section 6 — Evaluation, exports, and training curves

- evaluates best checkpoint on test split
- exports ONNX and INT8 ONNX
- plots training curves from persisted `history.csv`


In [ ]:
import matplotlib.pyplot as plt
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

best = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(best['model'], strict=False)
model.eval()

print('Evaluating on test split...')
test_metrics = eval_one_epoch(model, test_loader)
print('Test metrics:', test_metrics)

# Save clean PT weights (in addition to best.pt/last.pt trainer checkpoints)
pt_weights = EXPORT_DIR / 'yolopv2_finetuned.pt'
torch.save(model.state_dict(), pt_weights)

# Export ONNX (YOLOP forward: det_out, da_seg_out, ll_seg_out)
onnx_fp32 = EXPORT_DIR / 'yolopv2_finetuned.onnx'
onnx_int8 = EXPORT_DIR / 'yolopv2_finetuned_int8.onnx'

dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)

class ExportWrapper(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, images):
        det_out, da_out, ll_out = forward_unpack(self.m, images)
        return det_out, da_out, ll_out

export_model = ExportWrapper(model).to(device).eval()

with torch.no_grad():
    _ = export_model(dummy)

torch.onnx.export(
    export_model,
    dummy,
    str(onnx_fp32),
    opset_version=16,
    input_names=['images'],
    output_names=['det', 'drivable', 'lane'],
    do_constant_folding=True,
    dynamic_axes={'images': {0: 'batch'}},
    dynamo=False,
)

quantize_dynamic(
    model_input=str(onnx_fp32),
    model_output=str(onnx_int8),
    weight_type=QuantType.QInt8,
)

print('Saved artifacts:')
print(' - training checkpoints:', BEST_CKPT, 'and', LAST_CKPT)
print(' - weights:', pt_weights)
print(' - onnx fp32:', onnx_fp32)
print(' - onnx int8:', onnx_int8)

# Plot curves
hist = pd.read_csv(HISTORY_CSV)

plt.figure(figsize=(10, 5))
plt.plot(hist['epoch'], hist['train_total'], label='train_total')
plt.plot(hist['epoch'], hist['val_total'], label='val_total')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training curves (resumable / persisted)')
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(hist['epoch'], hist['train_lbox'], label='train_lbox')
plt.plot(hist['epoch'], hist['val_lbox'], label='val_lbox')
plt.plot(hist['epoch'], hist['train_lobj'], label='train_lobj')
plt.plot(hist['epoch'], hist['val_lobj'], label='val_lobj')
plt.plot(hist['epoch'], hist['train_lcls'], label='train_lcls')
plt.plot(hist['epoch'], hist['val_lcls'], label='val_lcls')
plt.plot(hist['epoch'], hist['train_lseg_da'], label='train_lseg_da')
plt.plot(hist['epoch'], hist['val_lseg_da'], label='val_lseg_da')
plt.plot(hist['epoch'], hist['train_lseg_ll'], label='train_lseg_ll')
plt.plot(hist['epoch'], hist['val_lseg_ll'], label='val_lseg_ll')
plt.plot(hist['epoch'], hist['train_liou_ll'], label='train_liou_ll')
plt.plot(hist['epoch'], hist['val_liou_ll'], label='val_liou_ll')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss components')
plt.grid(True)
plt.legend(ncol=3)
plt.show()
